# Opioid Use Disorder (OUD) Risk Prediction
## Exploratory notebook sandbox

This notebook is an exploratory sandbox for Machine Learning Operations (MLOps) development

Use it to
- Inspect raw and cleaned data
- Perform focused Exploratory Data Analysis (EDA) checks to validate assumptions
- Debug modules in `src/` step by step
- Run quick training and inference checks while iterating on features

Do not use it to
- Replace the reproducible pipeline in `src/main.py`
- Store production logic in notebook cells
- Write production artifacts to disk from the notebook

Canonical production entry point
- Run from terminal: `python -m src.main`

Why this separation matters
- The orchestrator (`src/main.py`) is the factory: deterministic inputs, deterministic outputs, clear provenance
- The notebook is the lab bench: interactive inspection, rapid iteration, visible intermediate states

### A) Environment setup and imports

Learning intent
- Make imports reliable regardless of how you opened Jupyter
- Ensure relative paths behave the same way for everyone in class
- Keep notebook code thin by calling functions from `src/` modules

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Repo root alignment
# Problem this solves
# - Notebooks can start with different working directories depending on IDE and settings
# - Relative paths then break unpredictably
#
# Approach
# - Search upward from the current folder until we find a folder that contains `src/`
# - Set that as the working directory so relative paths match the orchestrator behaviour
def find_repo_root(start: Path, marker_dir: str = "src", max_hops: int = 12) -> Path:
    current = start.resolve()
    for _ in range(max_hops):
        if (current / marker_dir).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise RuntimeError(
        f"Could not find repo root containing '{marker_dir}/' starting from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
os.chdir(PROJECT_ROOT)

# Ensure `import src...` works even if Jupyter started elsewhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())

# Imports from production modules
# Note: we intentionally do NOT import from src/main.py
from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference

### B) Sandbox configuration

Guideline
- Keep configuration in one place
- Use the same column names and defaults as the orchestrator where possible
- Prefer explicit lists over clever inference so errors are actionable

In [ ]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "opiod_raw_data.csv"

BINARY_SUM_COLS = [
    "A", "B", "C", "D", "E", "F",
    "H", "I", "J", "K", "L", "M", "N",
    "R", "S", "T",
    "Low_inc", "SURG",
]

SETTINGS = {
    "target_column": "OD",
    "problem_type": "classification",  # "classification" or "regression"
    "split": {"test_size": 0.05, "val_size": 0.15, "random_state": 42},
    "features": {
        "quantile_bin": ["rx_ds"],
        "categorical_onehot": [],
        "numeric_passthrough": [],
        "binary_sum_cols": BINARY_SUM_COLS,
        "n_bins": 4,
    },
    "validation": {
        "numeric_non_negative_cols": ["rx_ds"],
        # Teaching point
        # - Many production pipelines impute missing values in the feature pipeline
        # - This gate setting reflects that typical contract
        "check_missing_values": False,
    },
}

def three_way_split(
    X: pd.DataFrame,
    y: pd.Series,
    *,
    test_size: float,
    val_size: float,
    random_state: int,
    stratify: bool,
):
    """
    Sandbox copy aligned with the orchestrator intent

    Why it exists in the notebook
    - Beginners benefit from seeing the leakage gate explicitly
    - We avoid importing helpers from src/main.py to prevent side effects

    Behaviour
    - Try stratified splits for classification
    - If stratification fails, fall back to random splits with a clear message
    """
    from sklearn.model_selection import train_test_split

    if test_size <= 0 or val_size <= 0 or (test_size + val_size) >= 1.0:
        raise ValueError("Split sizes must satisfy 0 < test_size, 0 < val_size, and test_size + val_size < 1")

    stratify_y = y if stratify else None

    try:
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify_y,
        )

        relative_val_size = val_size / (1.0 - test_size)
        stratify_temp = y_temp if stratify else None

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=relative_val_size,
            random_state=random_state,
            stratify=stratify_temp,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

    except ValueError as e:
        print(f"[notebook] Stratified split failed: {e}")
        print("[notebook] Falling back to random split")

        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
        )

        relative_val_size = val_size / (1.0 - test_size)

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=relative_val_size,
            random_state=random_state,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("TARGET:", SETTINGS["target_column"])
print("PROBLEM_TYPE:", SETTINGS["problem_type"])

### 1) Load raw data (`src.load_data`)

Educational note
- We load raw data exactly once
- Raw data should be treated as immutable input

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw data not found at {RAW_DATA_PATH}\n"
        "Check that you opened the correct repository folder and that data/raw exists"
    )

df_raw = load_raw_data(RAW_DATA_PATH)
print("df_raw.shape:", df_raw.shape)
df_raw.head()

### 2) Focused EDA checks

Goal
- Validate assumptions before investing in feature engineering and training

Guideline
- Prefer small, targeted checks over long notebooks
- If an assumption is wrong, fix the pipeline, not the notebook

In [ ]:
print("Missing values (top 10 columns):")
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

target_col = SETTINGS["target_column"]
if target_col in df_raw.columns:
    vc = df_raw[target_col].value_counts(dropna=False)
    print("\nTarget distribution:")
    display(vc)
    if vc.sum() > 0:
        print("Positive class prevalence:", float(vc.get(1, 0) / vc.sum()))
else:
    print(f"\nTarget column '{target_col}' not found in raw data")

# Teaching variable: raw dataset may contain 'rx ds' or 'rx_ds'
rx_col = "rx_ds" if "rx_ds" in df_raw.columns else ("rx ds" if "rx ds" in df_raw.columns else None)
if rx_col is not None:
    print(f"\nSummary for '{rx_col}':")
    display(df_raw[rx_col].describe())

print("\nBinary indicator check (sample of 3 columns):")
for col in BINARY_SUM_COLS[:3]:
    if col in df_raw.columns:
        print(f"'{col}' unique values:", pd.Series(df_raw[col].unique()).head(10).tolist())

### 3) Clean data (`src.clean_data`)

Educational note
- Cleaning should be deterministic
- Cleaning should not learn from the data in a way that leaks information across splits

In [ ]:
df_clean = clean_dataframe(df_raw, target_column=SETTINGS["target_column"])
print("df_clean.shape:", df_clean.shape)
df_clean.head()

### 4) Didactic check: what changed after cleaning

Goal
- Make transformations visible
- Help you debug unexpected column name changes or dropped columns

In [ ]:
raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("Columns removed:")
display(removed_cols)

print("Columns added:")
display(added_cols)

print('Cleaned has "rx_ds":', "rx_ds" in df_clean.columns)

### 5) Validate data (security gate)

Educational note
- Validation is a fail-fast gate
- It prevents wasting time on training with broken assumptions

Guideline
- If validation fails, fix the upstream module or the data contract
- Do not patch around failures inside the notebook

In [ ]:
required_columns = (
    [SETTINGS["target_column"]]
    + SETTINGS["features"]["quantile_bin"]
    + SETTINGS["features"]["categorical_onehot"]
    + SETTINGS["features"]["numeric_passthrough"]
    + SETTINGS["features"]["binary_sum_cols"]
)
required_columns = list(dict.fromkeys(required_columns))

validate_dataframe(
    df=df_clean,
    required_columns=required_columns,
    check_missing_values=SETTINGS["validation"]["check_missing_values"],
    target_column=SETTINGS["target_column"],
    target_allowed_values=[0, 1] if SETTINGS["problem_type"] == "classification" else None,
    numeric_non_negative_cols=SETTINGS["validation"]["numeric_non_negative_cols"],
)

print("[notebook] Validation passed")

### 6) Build the feature recipe (`src.features`)

Educational note
- This step builds a preprocessing blueprint
- It must not fit on the full dataset in the notebook
- The recipe learns only when fitted on the training split inside the training pipeline

In [ ]:
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=SETTINGS["features"]["quantile_bin"],
    categorical_onehot_cols=SETTINGS["features"]["categorical_onehot"],
    numeric_passthrough_cols=SETTINGS["features"]["numeric_passthrough"],
    binary_sum_cols=SETTINGS["features"]["binary_sum_cols"],
    n_bins=SETTINGS["features"]["n_bins"],
)

print("[notebook] Feature recipe built, not fitted yet")
preprocessor

### 7) Leakage gate: three-way split (train, validation, test)

Educational note
- Train split is the only split allowed to learn preprocessing parameters and model weights
- Validation split is for iteration and model selection
- Test split is a final audit vault

In [ ]:
X = df_clean.drop(columns=[SETTINGS["target_column"]])
y = df_clean[SETTINGS["target_column"]]

if SETTINGS["problem_type"] == "classification" and y.nunique() < 2:
    raise ValueError(
        f"Target '{SETTINGS['target_column']}' has fewer than 2 classes after cleaning\n"
        "Classification training and stratified splitting cannot proceed"
    )

X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(
    X,
    y,
    test_size=SETTINGS["split"]["test_size"],
    val_size=SETTINGS["split"]["val_size"],
    random_state=SETTINGS["split"]["random_state"],
    stratify=(SETTINGS["problem_type"] == "classification"),
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

if len(X_test) == 0:
    raise ValueError("Test split is empty. Check split ratios and dataset size.")

### 8) Train (`src.train`)

Educational note
- This is the only step where `.fit()` happens
- The preprocessor and estimator are fitted only on training data

In [ ]:
model_pipeline = train_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    problem_type=SETTINGS["problem_type"],
)

print("[notebook] Training complete")
model_pipeline

### 9) Evaluate (`src.evaluate`)

Educational note
- We evaluate twice
  - Validation for development feedback
  - Test for audit-style confirmation

Metric note
- `evaluate_model` selects the metric based on `problem_type`
- Classification returns F1 score
- Regression returns root mean squared error (RMSE)

In [ ]:
val_metric = evaluate_model(
    model=model_pipeline,
    X_eval=X_val,
    y_eval=y_val,
    problem_type=SETTINGS["problem_type"],
)
print(f"[notebook] Validation metric: {val_metric:.4f}")

test_metric = evaluate_model(
    model=model_pipeline,
    X_eval=X_test,
    y_eval=y_test,
    problem_type=SETTINGS["problem_type"],
)
print(f"[notebook] Test metric: {test_metric:.4f}")

metric_name = "F1 score" if SETTINGS["problem_type"] == "classification" else "root mean squared error (RMSE)"
print("Metric meaning for this run:", metric_name)

### 10) Inference demo (`src.infer`)

Educational note
- Inference simulates what happens after training
- We deliberately use rows from the test split to simulate unseen cases

In [ ]:
sample_n = min(10, len(X_test))
X_infer_sample = X_test.sample(n=sample_n, random_state=SETTINGS["split"]["random_state"])

df_predictions = run_inference(
    model=model_pipeline,
    X_infer=X_infer_sample,
    include_proba=(SETTINGS["problem_type"] == "classification"),
)

print("[notebook] Inference results")
display(df_predictions.head(10))

### 11) Inspect production artifacts produced by the orchestrator

This notebook does not write artifacts to disk

Use this cell only after you run the orchestrator from terminal
- `python -m src.main`

Goal
- Prove that the factory output exists on disk
- Inspect outputs without modifying them

In [ ]:
from src.utils import load_model

CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "model.joblib"
PREDICTIONS_PATH = PROJECT_ROOT / "reports" / "predictions.csv"

try:
    clean_from_disk = pd.read_csv(CLEAN_DATA_PATH)
    preds_from_disk = pd.read_csv(PREDICTIONS_PATH)
    model_from_disk = load_model(MODEL_PATH)

    print("clean.csv shape:", clean_from_disk.shape)
    print("predictions.csv shape:", preds_from_disk.shape)
    print("loaded model type:", type(model_from_disk))

    display(preds_from_disk.head(10))

except Exception as e:
    print("Artifacts not found yet or could not be loaded")
    print("Run from terminal: python -m src.main")
    print("Error:", e)